# Web-Gold-40K — recovery-controlled smoke and mini

This separate notebook preserves `kaggle_gold.ipynb` and runs the registered `recovery_v1` experiment.

- **smoke**: 16 rows; causal inputs, shapes, finite loss/backward, and a real optimizer update.
- **mini**: 5,000 jointly stratified train rows plus the fixed v14 500-row validation subset for 5 epochs.
- Outputs: scalar CSV, detailed diagnostics JSON, experiment report, environment record, and all five epoch checkpoints.

The test split is never read. Run smoke first, inspect the report, then restart the kernel and change `STAGE` to `mini`.

In [ ]:
# 1. Pull modular code and record the exact environment.
from pathlib import Path
import importlib.metadata as metadata
import json
import os
import subprocess
import sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
SOURCE_ROOT = REPO_ROOT / 'src'

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'],
        check=True,
    )
else:
    subprocess.run(
        ['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)],
        check=True,
    )

requirements = [
    'transformers>=4.49,<5',
    'peft>=0.14,<1',
    'bitsandbytes>=0.45,<1',
    'accelerate>=1,<2',
    'scikit-learn>=1.4,<2',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements],
    check=True,
)

for module_name in list(sys.modules):
    if module_name == 'web_agent' or module_name.startswith('web_agent.'):
        del sys.modules[module_name]
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(REPO_ROOT)

packages = ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'scikit-learn']
environment = {name: metadata.version(name) for name in packages}
environment['python'] = sys.version.split()[0]
environment['git_commit'] = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
environment_path = Path('/kaggle/working/gold_recovery_v1_environment.json')
environment_path.write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))
print('Environment saved:', environment_path)

In [ ]:
# 2. Locate the attached dataset without downloading or extracting it.
EXPECTED_DATASET_ROOT = Path(
    '/kaggle/input/datasets/kiyasmahmud/web-gold-40k/final_data_set_40k'
)
ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
SPLIT_FILES = ('split_train.json', 'split_val.json', 'split_test.json')

def contains_splits(path: Path) -> bool:
    return path.is_dir() and all((path / name).is_file() for name in SPLIT_FILES)

def find_split_root() -> Path:
    if contains_splits(EXPECTED_DATASET_ROOT):
        return EXPECTED_DATASET_ROOT
    candidates = []
    if ATTACHED_ROOT.is_dir():
        for current, _, files in os.walk(ATTACHED_ROOT, followlinks=True):
            if set(SPLIT_FILES).issubset(files):
                candidates.append(Path(current))
    if not candidates:
        raise FileNotFoundError(
            'Attach kiyasmahmud/web-gold-40k; no structured split folder was found.'
        )
    return sorted(candidates, key=lambda path: (len(path.parts), str(path)))[0]

DATA_ROOT = find_split_root().resolve()
print('DATA_ROOT =', DATA_ROOT)
print('Dataset stays read-only under /kaggle/input.')

In [ ]:
# 3. Choose one stage. Smoke must pass before mini.
import torch
from web_agent.config import load_config
from web_agent.utils.seed import set_seed

STAGE = 'smoke'  # 'smoke' or 'mini'
SEED = 42
SMOKE_ROWS = 16
MINI_TRAIN_ROWS = 5_000
MINI_VAL_ROWS = 500
MINI_EPOCHS = 5

assert STAGE in {'smoke', 'mini'}
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
set_seed(SEED)

cfg = load_config('configs/backbones/qwen2vl_2b_gold.yaml')
cfg['data']['root'] = str(DATA_ROOT)
cfg['data']['causal_routing'] = True
cfg['data']['use_state_after'] = True
cfg['data']['use_visual_diff_text'] = False
cfg['data']['num_workers'] = 0 if STAGE == 'smoke' else 4

assert cfg['train']['controlled_experiment_tag'] == 'recovery_v1'
assert cfg['train']['early_stop_metric'] == 'outcome_mcc'
print('GPU:', torch.cuda.get_device_name(0))
print('STAGE:', STAGE, '| experiment:', cfg['train']['controlled_experiment_tag'])
print('pre-action  -> state_before + task/domain -> action, bbox, confidence-before')
print('post-action -> before + after + task/domain -> outcome, failure, recovery, memory')

In [ ]:
# 4. Inspect one causal batch before allocating the model.
from web_agent.data.gold_dataloader import build_gold_dataloader, load_gold_split
from web_agent.train.gold_stages import build_processor

processor = build_processor(cfg)
train_records = load_gold_split(cfg, 'train')
val_records = load_gold_split(cfg, 'val')
assert len(train_records) == 23_499, f'Unexpected train rows: {len(train_records)}'
assert len(val_records) == 7_861, f'Unexpected validation rows: {len(val_records)}'

inspection_loader = build_gold_dataloader(
    cfg, 'train', processor, records=train_records, limit=16, batch_size=4,
    shuffle=False, num_workers=0, seed=SEED, smoke=True,
)
inspection_batch = next(iter(inspection_loader))
required_streams = {
    'pre_input_ids', 'pre_pixel_values', 'pre_image_grid_thw',
    'post_input_ids', 'post_pixel_values', 'post_image_grid_thw',
}
assert required_streams.issubset(inspection_batch)
assert 'input_ids' not in inspection_batch, 'Shared stream would leak state_after.'
print('train rows:', len(train_records), '| validation rows:', len(val_records))
for key, value in inspection_batch.items():
    if torch.is_tensor(value):
        print(f'{key:28} shape={tuple(value.shape)} dtype={value.dtype}')

In [ ]:
# 5. Run the selected stage. Neither stage opens split_test.json.
from web_agent.train.gold_stages import run_gold_mini, run_gold_smoke
from web_agent.utils.results import save_mini_diagnostics_json, save_mini_result_csv

if STAGE == 'smoke':
    stage_report = run_gold_smoke(
        cfg, processor=processor, rows=SMOKE_ROWS, seed=SEED,
    )
else:
    stage_report = run_gold_mini(
        cfg, processor=processor, train_rows=MINI_TRAIN_ROWS,
        val_rows=MINI_VAL_ROWS, epochs=MINI_EPOCHS, seed=SEED,
    )

report_path = Path(f'/kaggle/working/gold_recovery_v1_{STAGE}_report.json')
report_path.write_text(json.dumps(stage_report, indent=2), encoding='utf-8')
result_csv_path = None
diagnostics_path = None
if STAGE == 'mini':
    result_csv_path = save_mini_result_csv(
        stage_report, '/kaggle/working/gold_recovery_v1_result.csv',
    )
    diagnostics_path = save_mini_diagnostics_json(
        stage_report, '/kaggle/working/gold_recovery_v1_diagnostics.json',
    )
print(json.dumps(stage_report, indent=2))
print('Report saved:', report_path)
if result_csv_path is not None:
    print('CSV saved:', result_csv_path)
    print('Diagnostics saved:', diagnostics_path)

In [ ]:
# 6. Enforce the engineering gate; report quality separately.
assert stage_report['status'] == 'PASS'

if STAGE == 'smoke':
    assert stage_report['dataset_rows'] == 16
    assert stage_report['processed_rows'] == 16
    assert stage_report['parameter_changed'] is True
    print('SMOKE PASSED. Restart, change STAGE to mini, then Run All.')
else:
    assert stage_report['train_rows'] == MINI_TRAIN_ROWS
    assert stage_report['val_rows'] == MINI_VAL_ROWS
    assert stage_report['test_rows_read'] == 0
    assert stage_report['loss_decreased'] is True
    assert stage_report['checkpoint_roundtrip'] is True
    assert len(stage_report['history']) == MINI_EPOCHS
    assert len(stage_report['epoch_checkpoints']) == MINI_EPOCHS
    assert stage_report['sampling']['duplicate_train_rows_scheduled'] == 0
    assert stage_report['sampling']['unique_train_rows_scheduled'] == MINI_TRAIN_ROWS
    assert stage_report['sampling']['validation_comparable_to_v14'] is True
    assert result_csv_path is not None and result_csv_path.is_file()
    assert diagnostics_path is not None and diagnostics_path.is_file()
    print('MINI ENGINEERING PASS; do not inspect test yet.')
    print('CONTROLLED QUALITY GATES:', stage_report['quality_gates']['status'])
    print(json.dumps(stage_report['quality_gates']['checks'], indent=2))
    print('Headline training remains blocked by review and full image hashing.')

## Interpretation rules

- Engineering PASS means the pipeline ran correctly; quality PASS means every predeclared comparison gate passed.
- Imbalanced recovery and memory require macro-F1, balanced accuracy, MCC, distributions, and confusion matrices—not accuracy alone.
- `recovery_outcome_*` is offline prediction on attempted rows, not executed browser recovery success.
- The validation rows and original training hyperparameters stay fixed against v14; only registered recovery training corrections change.
- Pillar 2 still requires multimodal ablation and has no standalone score.